In [ ]:
import sys
import os # File System Navigator

# Append the absolute path of the project root to Python's search map to enable 'src' imports
sys.path.append(os.path.abspath('..'))

from src.xor.dataset import get_dataloader
dataloader = get_dataloader(batch_size=4, shuffle=True)

# Grab one batch of data to test
X_batch, Y_batch = next(iter(dataloader))

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('..'))

from src.xor.TL_model import NeuronEnsemble
import numpy as np

num_in = 2
num_out = 10
test_ensemble = NeuronEnsemble(num_inputs=num_in, num_neurons=num_out)

sensory_inputs = np.random.randn(num_in)  # Example sensory inputs
print(f"sensory_inputs:", sensory_inputs)
c_n = np.random.randn(num_out)  # Example top-down apical inputs

r = test_ensemble.firing_rate(sensory_inputs, c_n)

sensory_inputs: [0.85904016 0.85512531]
Raw bottom-up input (z): [ 1.90302772 -0.79225898 -1.67947247  0.34544951  0.28826071  0.29688526
  0.98427133 -0.6172927   1.74387546  0.33455757]
Bottom-up input after leaky ReLU (phi_z): [ 1.90302772 -0.00792259 -0.01679472  0.34544951  0.28826071  0.29688526
  0.98427133 -0.00617293  1.74387546  0.33455757]
Bottom-up sensory activation (phi_z): [ 1.90302772 -0.00792259 -0.01679472  0.34544951  0.28826071  0.29688526
  0.98427133 -0.00617293  1.74387546  0.33455757]
Top-down apical activation (q_c): [0.15957995 0.40979114 0.70294654 0.34872797 0.32689693 0.3788262
 0.47830264 0.42964301 0.34947332 0.53270294]
Final firing rate (r): [ 2.20671278 -0.0111692  -0.02860052  0.46591742  0.38249225  0.40935318
  1.45505091 -0.00882508  2.35331342  0.51277737]


In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))

from src.xor.TL_model import NeuronEnsemble
from src.core.utils import test_function
import numpy as np

num_in = 2
num_out = 10
test_ensemble = NeuronEnsemble(num_inputs=num_in, num_neurons=num_out)

# ==========================================
# TEST SUITE 1: Leaky ReLU
# ==========================================
relu_tests = [
    # Test 1: Positive numbers should stay exactly the same
    ( {"x": np.array([5.0, 0.0]), "alpha": 0.01}, np.array([5.0, 0.0]) ),
    
    # Test 2: Negative numbers should be multiplied by the 0.01 leak
    ( {"x": np.array([-1.0, -100.0]), "alpha": 0.01}, np.array([-0.01, -1.0]) )
]

test_function(test_ensemble.leaky_relu, relu_tests)

# ==========================================
# TEST SUITE 2: Sensory Processing (Bottom-Up)
# ==========================================
# Math breakdown for Test 1:
# Input: [2.0, 1.0]
# z_n = (2.0*0.5 + 1.0*1.0) + 0.1 = 2.1  (Neuron 1)
# z_n = (2.0*-0.5 + 1.0*0.0) - 0.1 = -1.1 (Neuron 2)
# Apply Leaky ReLU to [2.1, -1.1] -> [2.1, -0.011]
sensory_tests = [
    ( {"sensory_inputs": np.array([2.0, 1.0])}, np.array([2.1, -0.011]) )
]

test_function(test_ensemble.sensory_proc, sensory_tests)

# ==========================================
# TEST SUITE 3: Dendritic Processing (Apical Sigmoid)
# ==========================================
dendritic_tests = [
    # Test 1: An input of 0 should perfectly center the sigmoid at 0.5
    ( {"c_n": np.array([0.0])}, np.array([0.5]) ),
    
    # Test 2: A huge positive number should hit the ceiling (1.0), 
    # and a huge negative number should hit the floor (0.0)
    ( {"c_n": np.array([100.0, -100.0])}, np.array([1.0, 0.0]) )
]

test_function(test_ensemble.dendritic_proc, dendritic_tests)

# ==========================================
# TEST SUITE 4: Total Firing Rate
# ==========================================
# Math breakdown for Test 1:
# phi_z (bottom up) = [2.1, -0.011]  (from sensory test)
# q_c (top down) = [0.5, 1.0]        (from dendritic test input [0.0, 100.0])
# r = (1.0 * q_c + 1) * phi_z
# Neuron 1: (0.5 + 1) * 2.1 = 3.15
# Neuron 2: (1.0 + 1) * -0.011 = -0.022
firing_tests = [
    ( 
      {"sensory_inputs": np.array([2.0, 1.0]), "c_n": np.array([0.0, 100.0]), "beta": 1.0}, 
      np.array([3.15, -0.022]) 
    )
]

test_function(test_ensemble.firing_rate, firing_tests)

--- Running tests for: leaky_relu ---
  [ERROR] Test 1 crashed with inputs {'x': array([5., 0.]), 'alpha': 0.01}. Error: unsupported format string passed to numpy.ndarray.__format__
  [ERROR] Test 2 crashed with inputs {'x': array([  -1., -100.]), 'alpha': 0.01}. Error: unsupported format string passed to numpy.ndarray.__format__
--- Results: 0 Passed | 2 Failed ---

--- Running tests for: sensory_proc ---
  [ERROR] Test 1 crashed with inputs {'sensory_inputs': array([2., 1.])}. Error: operands could not be broadcast together with shapes (10,) (2,) 
--- Results: 0 Passed | 1 Failed ---

--- Running tests for: dendritic_proc ---
  [ERROR] Test 1 crashed with inputs {'c_n': array([0.])}. Error: unsupported format string passed to numpy.ndarray.__format__
  [ERROR] Test 2 crashed with inputs {'c_n': array([ 100., -100.])}. Error: unsupported format string passed to numpy.ndarray.__format__
--- Results: 0 Passed | 2 Failed ---

--- Running tests for: firing_rate ---
  [ERROR] Test 1 crashe

False